## **Pulling the git repository**

In [ ]:
import os, sys

import sys
import os

REPO_URL  = "https://github.com/jacob2236/CSE5283-pose-estimation.git"
REPO_NAME = "CSE5283-pose-estimation"

if "google.colab" in sys.modules:
    print("Running in Google Colab")

    # If the repo doesn't exist, clone it
    if not os.path.exists(REPO_NAME) and os.path.basename(os.getcwd()) != REPO_NAME:
        !git clone -q {REPO_URL}
        os.chdir(REPO_NAME)
    else:
        # If it already exists, make sure directory is inside it and pull updates
        if os.path.basename(os.getcwd()) != REPO_NAME:
            os.chdir(REPO_NAME)
        print("Repo exists. Pulling latest changes...")
        !git pull

print("working directory:", os.getcwd())
files = os.listdir('.')
print(files)
print("camera.json present:", os.path.exists("camera.json"))


## **TASK 1**

In [ ]:
import json
import numpy as np

# Load the camera intrinsics from the JSON file
with open("camera.json", "r") as f:
    camera_data = json.load(f)

print(camera_data)

fx = camera_data["fx"]
fy = camera_data["fy"]
cx = camera_data["cx"]
cy = camera_data["cy"]
width  = camera_data.get("width")
height = camera_data.get("height")

# Build K explicitly (same convention as Lambda / mtx)
K = np.array([
    [fx,  0, cx],
    [ 0, fy, cy],
    [ 0,  0,  1]
])

print("Intrinsic matrix K =")
print(K)
print(f"fx = {fx}, fy = {fy}, cx = {cx}, cy = {cy}")
print(f"image size = {width} x {height}")

In [ ]:
import os
import glob
import cv2 as cv
import matplotlib.pyplot as plt

# Simplified version because there's just one jpg sitting next to the notebook
image_path = "0000_rgb1.jpg"

img = cv.imread(image_path)
img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(img_rgb)
plt.axis("off")
plt.title(os.path.basename(image_path))
plt.show()

Chosen Image

In [ ]:
#   1. Display the image.
#   2. Visually inspect it against the known HOPE object set.

plt.figure(figsize=(12, 9))
plt.imshow(img_rgb)
plt.axis("off")
plt.title("Inspect for HOPE objects")
plt.show()

# Manual list of objects found
identified_objects = [
    "BBQSauce",
    "Milk",
    "AlphabetSoup",
    "CreamCheese",
    "MacaroniAndChees",
    "Corn",
    "Butter",
    "OrangeJuice",
    "Pineapple",
    "Ketchup",

]

chosen_box_object = "MacaroniAndCheese"
print("Objects identified:", identified_objects)
print("Chosen box for homography:", chosen_box_object)

Chosen Image and objects within

## **TASK 2**

# **PART A**

In [ ]:
!pip install -q trimesh
import trimesh
import numpy as np

mesh = trimesh.load("obj_000012.ply")
print(mesh)
print("vertex count:", len(mesh.vertices))
print("units: millimeters (BOP/HOPE convention)")

# Measure via the axis-aligned bounding box of all vertices
bbox_min = mesh.vertices.min(axis=0)
bbox_max = mesh.vertices.max(axis=0)
extents  = bbox_max - bbox_min

print("bbox min:", np.round(bbox_min, 2))
print("bbox max:", np.round(bbox_max, 2))
print("extents (x, y, z) in mm:", np.round(extents, 2))

In [ ]:
FACE_NORMAL_AXIS = 0

span_axes = [a for a in range(3) if a != FACE_NORMAL_AXIS]
face_width  = extents[span_axes[0]]
face_height = extents[span_axes[1]]

print(f"face width  = {face_width:.2f} mm")
print(f"face height = {face_height:.2f} mm")

Set the normal face axis to +Z so the width and height spanned the direction x and y. I measured the mesh in millimeters.

## **PART B**

In [ ]:
# Corners of the chosen face, clicked in the image, in the SAME cyclic order as
# the model coordinates defined in Step 3
img_points = np.array([
    [867.,472.],
    [962.,894.],
    [1310.,814.],
    [1167.,415.],
], dtype=np.float64)

print("clicked pixel coordinates:")
for i, (u, v) in enumerate(img_points):
    print(f"  corner {i}: ({u:.1f}, {v:.1f})")

In [ ]:
labelled = img_rgb.copy()
for j, (u, v) in enumerate(img_points):
    labelled = cv.circle(labelled, (int(u), int(v)), 8, (255, 0, 0), -1)
    labelled = cv.putText(labelled, f'{j}', (int(u)+15, int(v)+10),
                          cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)

plt.figure(figsize=(10, 8))
plt.imshow(labelled); plt.axis("off")
plt.title("Clicked face corners")
plt.show()

My maually selected points in the image

## **PART C**

In [ ]:
model_points = np.array([
    [0.0,          0.0],
    [0.0,          face_height],
    [face_width,   face_height],
    [face_width,   0.0],
], dtype=np.float64)

image_height, image_width = img_rgb.shape[:2]

if width is not None and height is not None:
    assert (image_width, image_height) == (width, height), \
        f"camera.json says {width}x{height} but image is {image_width}x{image_height} — check you loaded the right camera.json / image pair"

print(f"Using K as-is from camera.json (no rescaling needed):\n{K}")

In [ ]:
H, _ = cv.findHomography(model_points, img_points, method=0)
print("H =\n", np.round(H, 4))

def factor_homography(Phi, K):
    """
    Factor a planar homography into pose. Returns lambda, the raw
    (not-yet-orthonormal) M = [r1 r2 r3], the corrected rotation Omega,
    translation tau, and the two Step-4 diagnostics.
    """
    K_inv = np.linalg.inv(K)
    Phi_tilde = K_inv @ Phi
    h1, h2, h3 = Phi_tilde[:, 0], Phi_tilde[:, 1], Phi_tilde[:, 2]

    lam = 2.0 / (np.linalg.norm(h1) + np.linalg.norm(h2))
    if (lam * h3)[2] < 0:          # enforce positive depth
        lam = -lam

    r1 = lam * h1
    r2 = lam * h2
    r3 = np.cross(r1, r2)
    tau = lam * h3

    M = np.column_stack([r1, r2, r3])

    ortho_error = np.linalg.norm(M.T @ M - np.eye(3), ord='fro')
    det_M = np.linalg.det(M)

    U, S, Vt = np.linalg.svd(M)
    Omega = U @ np.diag([1.0, 1.0, np.linalg.det(U @ Vt)]) @ Vt
    correction_norm = np.linalg.norm(M - Omega, ord='fro')

    return {
        "lambda": lam, "M_raw": M, "Omega": Omega, "tau": tau,
        "ortho_error_MtM_minus_I": ortho_error,
        "det_M": det_M,
        "correction_norm_M_minus_Omega": correction_norm,
    }

result = factor_homography(H, K)
lam, Omega, tau = result["lambda"], result["Omega"], result["tau"]

print(f"lambda = {lam:.6f}")
print("R (Omega) =\n", np.round(Omega, 4))
print("t (tau) =\n", np.round(tau, 4))

units in millimeters.

## **PART D**

In [ ]:
print(f"||M^T M - I||_F (raw M, before orthonormalization) = {result['ortho_error_MtM_minus_I']:.6f}")
print(f"det(M) (raw M, before orthonormalization)           = {result['det_M']:.6f}")
print(f"||M - R||_F (after SVD correction)                  = {result['correction_norm_M_minus_Omega']:.6f}")

The small value of ‖M − R‖_F = 0.0177 indicates that my four clicked corner points were geometrically consistent and accurately localized, since a nearly-rigid raw estimate (needing only a tiny SVD correction to become a true rotation) is what you'd expect from clean, precise correspondences rather than noisy or mislabeled clicks.

## **PART E**

In [ ]:
u_axis, v_axis = span_axes      # from Step 1
n_axis = FACE_NORMAL_AXIS

FACE_OFFSET = bbox_max[FACE_NORMAL_AXIS]

def to_face_frame(vertices):
    verts = np.asarray(vertices, dtype=np.float64)
    X = verts[:, u_axis] - bbox_min[u_axis]
    Y = verts[:, v_axis] - bbox_min[v_axis]
    Z = verts[:, n_axis] - FACE_OFFSET
    return np.column_stack([X, Y, Z])

world_vertices = to_face_frame(mesh.vertices)

def project_points(world_pts, Omega, tau, K):
    world_pts = np.asarray(world_pts, dtype=np.float64).reshape(-1, 3)
    cam_pts = (Omega @ world_pts.T).T + tau.reshape(1, 3)
    proj = (K @ cam_pts.T).T
    return proj[:, :2] / proj[:, 2:3]

model_points_3d = np.column_stack([model_points, np.zeros(4)])
reproj_uv = project_points(model_points_3d, Omega, tau, K)

errors = np.linalg.norm(reproj_uv - img_points, axis=1)
for i, (pt, err) in enumerate(zip(reproj_uv, errors)):
    print(f"corner {i}: reprojected=({pt[0]:.1f},{pt[1]:.1f})  "
          f"clicked=({img_points[i][0]:.1f},{img_points[i][1]:.1f})  error={err:.2f}px")
print(f"mean reprojection error: {errors.mean():.2f} px")

In [ ]:
edges = mesh.edges_unique
proj_vertices = project_points(world_vertices, Omega, tau, K)

wireframe_img = img_rgb.copy()
for i, j in edges:
    p1 = tuple(np.round(proj_vertices[i]).astype(int))
    p2 = tuple(np.round(proj_vertices[j]).astype(int))
    wireframe_img = cv.line(wireframe_img, p1, p2, (0, 255, 0), 2)

for u, v in img_points:
    wireframe_img = cv.circle(wireframe_img, (int(u), int(v)), 8, (255, 0, 0), -1)

plt.figure(figsize=(10, 8))
plt.imshow(wireframe_img); plt.axis("off")
plt.title("Reprojected wireframe using recovered pose")
plt.show()

Reprojected mesh on top of the selected points in the image

In [ ]:
print("camera_data keys:", camera_data.keys())
print("width/height from json:", width, height)
print("actual image shape:", img_rgb.shape)
print("K =\n", K)
print("face_width =", face_width, " face_height =", face_height)

## **TASK 3**

## **PART A**

In [ ]:
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

# Guard check and state setup
# Ensures required mesh and camera structures exist
mesh_t2 = mesh
K_int = np.asarray(K, dtype=np.float64)

if "width" in locals() and width is not None:
    width_i, height_i = int(width), int(height)
else:
    height_i, width_i = img_rgb.shape[:2]

# Ensure bounding box extents are available
bbox_min = mesh_t2.vertices.min(axis=0)
bbox_max = mesh_t2.vertices.max(axis=0)
object_center = (bbox_min + bbox_max) / 2.0

# Rasterizer Function
def rasterize_mesh(mesh_obj, R, t, K_mat, w, h, light_dir=np.array([0.3, -0.3, -1.0])):
    """
    Project mesh faces with (R, t, K), sort back-to-front, and rasterize flat-shaded triangles.
    Returns:
        rgb: (H, W, 3) uint8 image
        depth: (H, W) float32 depth map in mesh units (mm)
    """
    verts = np.asarray(mesh_obj.vertices, dtype=np.float64)
    faces = np.asarray(mesh_obj.faces, dtype=np.int64)

    # Transform to camera space
    cam_pts = (R @ verts.T).T + t.reshape(1, 3)
    proj = (K_mat @ cam_pts.T).T
    pix = proj[:, :2] / proj[:, 2:3]
    depth_per_vert = cam_pts[:, 2]

    # Face normals for directional shading
    if hasattr(mesh_obj, "face_normals") and mesh_obj.face_normals is not None:
        face_normals = mesh_obj.face_normals
    else:
        v0, v1, v2 = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
        face_normals = np.cross(v1 - v0, v2 - v0)
        face_normals /= (np.linalg.norm(face_normals, axis=1, keepdims=True) + 1e-9)

    light_dir_unit = light_dir / np.linalg.norm(light_dir)
    shade = np.clip(face_normals @ (-light_dir_unit), 0.2, 1.0)

    face_depth = depth_per_vert[faces].mean(axis=1)
    order = np.argsort(-face_depth)

    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    depth = np.zeros((h, w), dtype=np.float32)
    base_color = np.array([160, 170, 190])

    for f_idx in order:
        i, j, k = faces[f_idx]
        d = face_depth[f_idx]
        if d <= 0:
            continue
        tri = pix[[i, j, k]].astype(np.int32)
        if np.any(tri[:, 0] < -w) or np.any(tri[:, 0] > 2 * w):
            continue
        color = tuple(int(c * shade[f_idx]) for c in base_color)
        cv.fillConvexPoly(rgb, tri, color)
        cv.fillConvexPoly(depth, tri, float(d))

    return rgb, depth

# Compute Camera Pose
distance_template = 380.0
azim_deg = 15.0
elev_deg = -18.0   # Negative elevation places camera below the center to expose bottom face

azim = np.radians(azim_deg)
elev = np.radians(elev_deg)

cam_pos = object_center + distance_template * np.array([
    np.cos(elev) * np.cos(azim),
    np.cos(elev) * np.sin(azim),
    np.sin(elev)
])

forward = object_center - cam_pos
forward = forward / np.linalg.norm(forward)

world_up = np.array([0.0, 0.0, 1.0])
right = np.cross(forward, world_up)
right = right / np.linalg.norm(right)

cam_down = np.cross(forward, right)
cam_down = cam_down / np.linalg.norm(cam_down)

R_template = np.vstack([right, cam_down, forward])
t_template = -R_template @ cam_pos

# Render RGB & Depth
rgb_template, depth_template = rasterize_mesh(
    mesh_t2, R_template, t_template, K_int, width_i, height_i
)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(rgb_template)
plt.axis("off")
plt.title("Template Render (Upright Front + Bottom View)")

plt.subplot(1, 2, 2)
valid_mask = depth_template > 0
depth_vis = depth_template.copy()
depth_vis[~valid_mask] = np.nan
plt.imshow(depth_vis, cmap="plasma")
plt.axis("off")
plt.title("Template Depth (mm)")
plt.colorbar(label="Depth (mm)")
plt.tight_layout()
plt.show()

print(f"Render complete: {width_i}x{height_i}")
print(f"Depth range on object: {depth_template[valid_mask].min():.2f} mm to {depth_template[valid_mask].max():.2f} mm")

Template render of mesh and depth map of mesh

## **PART B**

In [ ]:
# Combine Automated Corners (0-3) + Manual Features (4-5)

import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

# Automated Corners 0-3 (First Plane)
pts_model_4 = np.array([
    [0.0, 0.0],                # Pt 0: Top-Left
    [0.0, face_height],        # Pt 1: Bottom-Left
    [face_width, face_height], # Pt 2: Bottom-Right
    [face_width, 0.0],         # Pt 3: Top-Right
], dtype=np.float64)

# Project to Scene Photo via Task 2 Homography
pts_model_homog = np.column_stack([pts_model_4, np.ones(4)])
scene_corners_homog = (H @ pts_model_homog.T).T
scene_corners_4 = scene_corners_homog[:, :2] / scene_corners_homog[:, 2:3]

# Project to Template Render via 3D Mesh geometry
pts_mesh_4d = np.zeros((4, 3), dtype=np.float64)
pts_mesh_4d[:, 0] = FACE_OFFSET
pts_mesh_4d[:, 1] = bbox_min[1] + pts_model_4[:, 0]
pts_mesh_4d[:, 2] = bbox_max[2] - pts_model_4[:, 1]

cam_pts_tpl_4 = (R_template @ pts_mesh_4d.T).T + t_template.reshape(1, 3)
proj_tpl_4 = (K_int @ cam_pts_tpl_4.T).T
tpl_corners_4 = proj_tpl_4[:, :2] / proj_tpl_4[:, 2:3]

# Pull the 4 corners 6% inward toward their centroid so they sit safely on the mesh surface
center_tpl_corners = tpl_corners_4.mean(axis=0)
for k in range(4):
    tpl_corners_4[k] += 0.06 * (center_tpl_corners - tpl_corners_4[k])

# Manual Points 4 and 5
manual_template_pts = np.array([
    [1210.96, 825.90],   # Bottom right ob bottom of the box
    [750.82, 840.14],  # Bottom left of the bottom of the box
], dtype=np.float64)

manual_scene_pts = np.array([
    [1290.0, 850.0],   # Bottom right of bottom
    [959.0, 919.0],  # Bottom left of bottom
], dtype=np.float64)

# Stack into the Final 6-Point Arrays
pts_template_rendered = np.vstack([tpl_corners_4, manual_template_pts])
pts_scene_photo       = np.vstack([scene_corners_4, manual_scene_pts])

print(f"{'Idx':>3} | {'Template Render (u, v)':>22} | {'Scene Photo (u, v)':>22}")
print("-" * 55)
for idx, (p_t, p_s) in enumerate(zip(pts_template_rendered, pts_scene_photo)):
    source = "Automated Corner" if idx < 4 else "Manual Feature"
    print(f"{idx:3d} | ({p_t[0]:7.2f}, {p_t[1]:7.2f})       | ({p_s[0]:7.2f}, {p_s[1]:7.2f})   [{source}]")

# Side-by-Side Visualization
h_tpl, w_tpl = rgb_template.shape[:2]
h_scn, w_scn = img_rgb.shape[:2]
vis_canvas = np.zeros((max(h_tpl, h_scn), w_tpl + w_scn, 3), dtype=np.uint8)
vis_canvas[:h_tpl, :w_tpl] = rgb_template
vis_canvas[:h_scn, w_tpl:w_tpl+w_scn] = img_rgb

colors = plt.cm.tab10(np.linspace(0, 1, 6))[:, :3] * 255

for idx, (p_t, p_s) in enumerate(zip(pts_template_rendered, pts_scene_photo)):
    pt_a = (int(round(p_t[0])), int(round(p_t[1])))
    pt_b = (int(round(p_s[0])) + w_tpl, int(round(p_s[1])))
    col = tuple(int(c) for c in colors[idx])

    cv.circle(vis_canvas, pt_a, 8, col, -1)
    cv.circle(vis_canvas, pt_b, 8, col, -1)
    cv.line(vis_canvas, pt_a, pt_b, col, 2)
    cv.putText(vis_canvas, str(idx), (pt_a[0] - 25, pt_a[1] - 10), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv.putText(vis_canvas, str(idx), (pt_b[0] + 15, pt_b[1] - 10), cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

plt.figure(figsize=(16, 9))
plt.imshow(vis_canvas)
plt.axis("off")
plt.title("Matched Correspondences: Template Render (Left) vs. Scene Photo (Right)")
plt.show()

Mathcing the selected points from the template in 3D to the real image with 6 points, 4 on one plane and the 5 + 6 on a second plane

## **PART C**

In [ ]:
# Unproject Template Pixels to 3-D
def sample_valid_depth(depth_img, u, v, search_radius=6):
    H_img, W_img = depth_img.shape
    u_c, v_c = int(round(u)), int(round(v))

    if 0 <= v_c < H_img and 0 <= u_c < W_img and depth_img[v_c, u_c] > 0:
        return float(depth_img[v_c, u_c])

    u0 = int(np.floor(u))
    u1 = min(u0 + 1, W_img - 1)
    v0 = int(np.floor(v))
    v1 = min(v0 + 1, H_img - 1)
    du, dv = u - u0, v - v0

    d = (1 - du) * (1 - dv) * depth_img[v0, u0] + \
        du * (1 - dv) * depth_img[v0, u1] + \
        (1 - du) * dv * depth_img[v1, u0] + \
        du * dv * depth_img[v1, u1]

    if d > 0:
        return float(d)

    # Search local window if on silhouette boundary
    v_min, v_max = max(0, v_c - search_radius), min(H_img, v_c + search_radius + 1)
    u_min, u_max = max(0, u_c - search_radius), min(W_img, u_c + search_radius + 1)
    patch = depth_img[v_min:v_max, u_min:u_max]
    valid_vals = patch[patch > 0]
    if len(valid_vals) > 0:
        return float(np.median(valid_vals))
    return 0.0

fx = K_int[0, 0]
fy = K_int[1, 1]
cx = K_int[0, 2]
cy = K_int[1, 2]

X_cam_list = []
X_mesh_list = []

for idx, (u, v) in enumerate(pts_template_rendered):
    Z = sample_valid_depth(depth_template, u, v)
    assert Z > 0, f"Point {idx} at ({u:.2f}, {v:.2f}) sampled invalid background depth (0.0)."

    Xc = (u - cx) * Z / fx
    Yc = (v - cy) * Z / fy
    Zc = Z
    pt_cam = np.array([Xc, Yc, Zc], dtype=np.float64)
    X_cam_list.append(pt_cam)

    # Invert template camera pose to recover canonical mesh coordinates
    pt_mesh = R_template.T @ (pt_cam - t_template)
    X_mesh_list.append(pt_mesh)

X_cam_arr = np.array(X_cam_list)
X_mesh_arr = np.array(X_mesh_list)

print("--- 3 Unprojected Template Camera Points (X_cam) ---")
for i in range(3):
    print(f"Point {i}: X_cam = [{X_cam_arr[i,0]:.2f}, {X_cam_arr[i,1]:.2f}, {X_cam_arr[i,2]:.2f}] (units: mm)")

print("\nUnits Confirmation: Millimeters (inherited from mesh_t2 vertex definitions and depth rasterization).")

## **PART D**

In [ ]:
# Solve PnP and Report Error
success, rvec, tvec = cv.solvePnP(
    objectPoints=X_mesh_arr.reshape(-1, 1, 3),
    imagePoints=pts_scene_photo.reshape(-1, 1, 2),
    cameraMatrix=K_int,
    distCoeffs=None,
    flags=cv.SOLVEPNP_ITERATIVE
)

assert success, "PnP solver failed to converge!"

R_pnp, _ = cv.Rodrigues(rvec)
t_pnp = tvec.flatten()

reprojected_pts, _ = cv.projectPoints(
    X_mesh_arr.reshape(-1, 1, 3), rvec, tvec, K_int, None
)
reprojected_pts = reprojected_pts.reshape(-1, 2)

errors_pnp = np.linalg.norm(reprojected_pts - pts_scene_photo, axis=1)
mean_reproj_err = np.mean(errors_pnp)

print("Recovered R (PnP) =\n", np.round(R_pnp, 4))
print("Recovered t (PnP) =\n", np.round(t_pnp, 4), "(mm)")
print(f"Mean Reprojection Error: {mean_reproj_err:.3f} px")

## **PART E**

In [ ]:
# Wireframe Overlay on Real Photo
cam_pts_scene = (R_pnp @ mesh_t2.vertices.T).T + t_pnp.reshape(1, 3)
proj_scene = (K_int @ cam_pts_scene.T).T
mesh_uv_scene = proj_scene[:, :2] / proj_scene[:, 2:3]

overlay_img = img_rgb.copy()
edges = mesh_t2.edges_unique

for i, j in edges:
    pt1 = tuple(np.round(mesh_uv_scene[i]).astype(int))
    pt2 = tuple(np.round(mesh_uv_scene[j]).astype(int))
    cv.line(overlay_img, pt1, pt2, (0, 255, 0), 2)

for u, v in pts_scene_photo:
    cv.circle(overlay_img, (int(round(u)), int(round(v))), 7, (255, 0, 0), -1)

for u, v in reprojected_pts:
    cv.circle(overlay_img, (int(round(u)), int(round(v))), 4, (0, 0, 255), -1)

plt.figure(figsize=(12, 9))
plt.imshow(overlay_img)
plt.axis("off")
plt.title("Wireframe Overlay from PnP Pose (Green) with Targets (Blue) vs Reprojections (Red)")
plt.show()

3D repojection of mesh onto the real image. The mesh seems to fit very well towards the bottom of the object where I was able to select 2 additional points on a different plane than the rest of the points, giving that accurate depth and 3D matching. But is a little off towards the top right corner, almost making the mesh look like it is closer to the camera than the real box. This most likely happens because of the fact there is only one plane depctied in the points chosen towards this edge of the box object.

## **PART F**

In [ ]:
# Cell 6: Coordinate Alignment & Pose Comparison
# Transform matrix from mesh coordinates to face coordinates (Task 2 setup):
P_mat = np.zeros((3, 3))
P_mat[0, u_axis] = 1.0
P_mat[1, v_axis] = -1.0
P_mat[2, n_axis] = 1.0

c0 = np.zeros(3)
c0[u_axis] = bbox_min[u_axis]
c0[v_axis] = bbox_max[v_axis]
c0[n_axis] = FACE_OFFSET

# Task 2 pose expressed in the canonical mesh frame:
R_task2_mesh = Omega @ P_mat
t_task2_mesh = tau - R_task2_mesh @ c0

# Task 3 pose from PnP directly maps canonical mesh frame -> camera frame:
R_task3_mesh = R_pnp
t_task3_mesh = t_pnp

# Rotation difference angle theta:
# theta = arccos( (trace(R1 @ R2.T) - 1) / 2 )
R_diff = R_task3_mesh @ R_task2_mesh.T
trace_val = np.clip((np.trace(R_diff) - 1.0) / 2.0, -1.0, 1.0)
theta_rad = np.arccos(trace_val)
theta_deg = np.degrees(theta_rad)

# Translation difference in mesh units (mm):
t_diff = np.linalg.norm(t_task3_mesh - t_task2_mesh)

print("--- Comparison of Task 2 (Homography) vs Task 3 (PnP) ---")
print(f"Rotation difference angle: {theta_deg:.3f} degrees ({theta_rad:.4f} rad)")
print(f"Translation difference:    {t_diff:.2f} mm")
print(f"t (Task 2 aligned): {np.round(t_task2_mesh, 2)}")
print(f"t (Task 3 PnP):     {np.round(t_task3_mesh, 2)}")

## **TASK 4**

In [ ]:
import os, sys, glob
REPO = "/content/mast3r_repo"
if not os.path.exists(REPO):
    !git clone -q --recursive https://github.com/naver/mast3r {REPO}
    !pip install -q -r {REPO}/requirements.txt -r {REPO}/dust3r/requirements.txt

for m in [k for k in sys.modules if k.startswith(("mast3r", "dust3r"))]:
    del sys.modules[m]
sys.path = [p for p in sys.path if p not in ("", os.getcwd())]   # avoid a local ./mast3r folder shadowing
sys.path.insert(0, REPO); sys.path.insert(0, REPO + "/dust3r")
print("cwd mast3r folder?", os.path.exists("mast3r"), "| repo has model.py:", os.path.exists(REPO + "/mast3r/model.py"))
import mast3r.model; print("mast3r.model OK from", mast3r.__file__)
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
SIZE = 512

## **PART A**

In [ ]:
# ===== Task 4, Cell 1 (revised): MASt3R matching on object crops =====
import os, numpy as np, cv2 as cv, matplotlib.pyplot as plt, torch
from PIL import Image
from mast3r.model import AsymmetricMASt3R
from mast3r.fast_nn import fast_reciprocal_NNs
import mast3r.utils.path_to_dust3r
from dust3r.inference import inference
from dust3r.utils.image import load_images

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
SIZE = 512
print("device:", dev)


SCENE_BOX = None

if 'model' not in globals() or next(model.parameters()).device.type != dev:
    model = AsymmetricMASt3R.from_pretrained(
        "naver/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric").to(dev)

h_t, w_t = rgb_template.shape[:2]; h_s, w_s = img_rgb.shape[:2]

# Template crop: tight box around rendered object (depth > 0) + margin
ys, xs = np.where(depth_template > 0)
m = int(0.10 * max(xs.max()-xs.min(), ys.max()-ys.min()))
tbox = (max(xs.min()-m, 0), max(ys.min()-m, 0), min(xs.max()+m, w_t), min(ys.max()+m, h_t))
sbox = SCENE_BOX if SCENE_BOX is not None else (0, 0, w_s, h_s)

crop_t = np.ascontiguousarray(np.asarray(rgb_template, np.uint8)[tbox[1]:tbox[3], tbox[0]:tbox[2]])
crop_s = np.ascontiguousarray(np.asarray(img_rgb, np.uint8)[sbox[1]:sbox[3], sbox[0]:sbox[2]])

os.makedirs("mast3r_images", exist_ok=True)
Image.fromarray(crop_t).save("mast3r_images/template.png")
Image.fromarray(crop_s).save("mast3r_images/scene.png")
images = load_images(["mast3r_images/template.png", "mast3r_images/scene.png"], size=SIZE, square_ok=True)

out = inference([tuple(images)], model, dev, batch_size=1, verbose=False)
d1 = out['pred1']['desc'].squeeze(0).detach()
d2 = out['pred2']['desc'].squeeze(0).detach()
m_t, m_s = fast_reciprocal_NNs(d1, d2, subsample_or_initxy1=4, device=dev,
                               dist='dot', block_size=2**13)
m_t, m_s = np.asarray(m_t, float), np.asarray(m_s, float)
print(f"Raw reciprocal NN matches: {len(m_t)}")

# MASt3R resizes the crop (long side=SIZE) then center-crops; map back to FULL-image pixels
def to_orig(pts, crop_w, crop_h, true_shape, box):
    W1, H1 = round(crop_w*SIZE/max(crop_w, crop_h)), round(crop_h*SIZE/max(crop_w, crop_h))
    cx_, cy_ = W1//2, H1//2
    hw, hh = ((2*cx_)//16)*8, ((2*cy_)//16)*8
    assert (2*hw, 2*hh) == (true_shape[1], true_shape[0]), "crop mismatch"
    return (pts + [cx_-hw, cy_-hh]) * [crop_w/W1, crop_h/H1] + [box[0], box[1]]

uv_t = to_orig(m_t, crop_t.shape[1], crop_t.shape[0], images[0]['true_shape'][0], tbox)
uv_s = to_orig(m_s, crop_s.shape[1], crop_s.shape[0], images[1]['true_shape'][0], sbox)

# display random subset of lines
canvas = np.zeros((max(h_t, h_s), w_t+w_s, 3), np.uint8)
canvas[:h_t, :w_t] = rgb_template; canvas[:h_s, w_t:] = img_rgb
for bx, off in ((tbox, 0), (sbox, w_t)):   # draw the crop boxes MASt3R actually saw
    cv.rectangle(canvas, (bx[0]+off, bx[1]), (bx[2]+off, bx[3]), (255, 255, 0), 2)
rng = np.random.default_rng(0)
for i in rng.choice(len(uv_t), min(100, len(uv_t)), replace=False):
    col = tuple(int(c) for c in rng.integers(50, 255, 3))
    a = (int(uv_t[i,0]), int(uv_t[i,1])); b = (int(uv_s[i,0])+w_t, int(uv_s[i,1]))
    cv.circle(canvas, a, 5, col, -1); cv.circle(canvas, b, 5, col, -1); cv.line(canvas, a, b, col, 1)
plt.figure(figsize=(16, 8)); plt.imshow(canvas); plt.axis('off')
plt.title(f"MASt3R matches ({len(uv_t)} raw, 100 shown; yellow = crops fed to MASt3R)"); plt.show()

Mast3r matching points between the template and the real image

## **PART B + C**

In [ ]:
# unproject -> PnP RANSAC -> compare
RANSAC_THRESH = 8.0   # px, tune for your image resolution

# Keep only matches whose template pixel lies on the object (depth > 0)
ui, vi = np.round(uv_t[:,0]).astype(int), np.round(uv_t[:,1]).astype(int)
inb = (ui>=0)&(ui<w_t)&(vi>=0)&(vi<h_t)
Z = np.zeros(len(uv_t)); Z[inb] = depth_template[vi[inb], ui[inb]]
keep = Z > 0
print(f"Matches with valid template depth: {keep.sum()} / {len(uv_t)}")

Zk, uk, vk = Z[keep], uv_t[keep,0], uv_t[keep,1]
X_cam = np.stack([(uk-K_int[0,2])*Zk/K_int[0,0], (vk-K_int[1,2])*Zk/K_int[1,1], Zk], 1)
X_mesh = (R_template.T @ (X_cam - t_template).T).T
pts_img = uv_s[keep]

ok, rv, tv, inl = cv.solvePnPRansac(
    X_mesh.reshape(-1,1,3), pts_img.reshape(-1,1,2), K_int, None,
    iterationsCount=5000, reprojectionError=RANSAC_THRESH,
    confidence=0.999, flags=cv.SOLVEPNP_AP3P)
assert ok and inl is not None, "RANSAC failed"
inl = inl.ravel()
rv, tv = cv.solvePnPRefineLM(X_mesh[inl].reshape(-1,1,3), pts_img[inl].reshape(-1,1,2),
                             K_int, None, rv, tv)          # polish on inliers only
R4, _ = cv.Rodrigues(rv); t4 = tv.ravel()

proj, _ = cv.projectPoints(X_mesh[inl].reshape(-1,1,3), rv, tv, K_int, None)
err = np.linalg.norm(proj.reshape(-1,2) - pts_img[inl], axis=1)
print(f"Inliers: {len(inl)} / {len(X_mesh)}")
print("R (Task 4) =\n", np.round(R4, 4)); print("t (Task 4) =", np.round(t4, 2), "mm")
print(f"Inlier reprojection error: mean {err.mean():.3f} px, median {np.median(err):.3f} px")

# compare with Task 3
tr = np.clip((np.trace(R4 @ R_pnp.T) - 1)/2, -1, 1)
print("\n--- Task 4 (MASt3R+RANSAC) vs Task 3 (hand-matched PnP) ---")
print(f"Rotation diff: {np.degrees(np.arccos(tr)):.3f} deg | Translation diff: {np.linalg.norm(t4 - t_pnp):.2f} mm")
tr2 = np.clip((np.trace(R4 @ R_task2_mesh.T) - 1)/2, -1, 1)
print(f"(Extra) vs Task 2: {np.degrees(np.arccos(tr2)):.3f} deg, {np.linalg.norm(t4 - t_task2_mesh):.2f} mm")
print(f"Points used -> Task 3: {len(pts_scene_photo)} | Task 4: {len(X_mesh)} fed to RANSAC, {len(inl)} inliers")

_, d_scn = rasterize_mesh(mesh_t2, R4, t4, K_int, w_s, h_s)
us, vs = np.clip(np.round(pts_img[:,0]).astype(int), 0, w_s-1), np.clip(np.round(pts_img[:,1]).astype(int), 0, h_s-1)
on_obj = d_scn[vs, us] > 0
print(f"\nScene matches inside projected silhouette: all {on_obj.mean()*100:.1f}% | inliers {on_obj[inl].mean()*100:.1f}%")

## **TASK D**

The template render is untextured, flat-shaded and on an empty background. That could bias MASt3R toward matching the box's silhouette and edges rather than its printed logo and text. In the photo those edges can also be matched to background clutter, such as table edges, shadows or other objects. This would give confident but wrong correspondences that sit systematically off the object. A second risk is that the render has no lighting or texture cues, so matches may cluster on high-contrast borders instead of being spread across the box face.

Check: the last lines of Cell 2 project the recovered mesh into the photo and count the fraction of scene-side matches that fall inside its silhouette. A low fraction means background-driven matches. You can also compare the inlier and outlier match locations. A stronger version masks the object out of the photo, pastes it onto a blank background, re-runs MASt3R and compares the pose. A large shift would show the background was biasing the matches.